# 01 — Exploratory Data Analysis (RSNA Bone Age)

In [ ]:
import sys; sys.path.append("../src")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import config
sns.set_theme(style="whitegrid")
df = pd.read_csv(config.TRAIN_CSV)
print(df.shape); df.head()

In [ ]:
df.describe(include="all")

## Bone age distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["boneage"], bins=50, ax=axes[0]); axes[0].set(title="Bone age (months)", xlabel="months")
sns.boxplot(data=df, x="male", y="boneage", ax=axes[1]); axes[1].set(title="Bone age by sex", xticks=[0, 1], xticklabels=["female", "male"])
plt.show()
print(df.groupby("male")["boneage"].describe().round(1))

## Sex balance

In [ ]:
print(df["male"].value_counts(normalize=True).rename({True: "male", False: "female"}).round(3))

## Image sizes

In [ ]:
sizes = np.array([Image.open(config.TRAIN_IMG_DIR / f"{i}.png").size for i in df["id"]])
df["width"], df["height"] = sizes[:, 0], sizes[:, 1]
print(df[["width", "height"]].describe().round(0))
print("aspect ratio (h/w):", (df.height / df.width).describe().round(2).to_dict())
sns.jointplot(data=df, x="width", y="height", s=8, alpha=0.3, height=5); plt.show()

## Sample images across ages

In [ ]:
picks = df.sort_values("boneage").iloc[np.linspace(0, len(df) - 1, 12).astype(int)]
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
    ax.imshow(cv2.imread(str(config.TRAIN_IMG_DIR / f"{r.id}.png"), cv2.IMREAD_GRAYSCALE), cmap="gray")
    ax.set_title(f"{r.boneage} mo ({r.boneage / 12:.1f} y), {'M' if r.male else 'F'}"); ax.axis("off")
plt.tight_layout(); plt.show()

## Image quality: brightness and contrast

In [ ]:
sample = df.sample(1000, random_state=config.SEED).copy()
imgs = [cv2.imread(str(config.TRAIN_IMG_DIR / f"{i}.png"), cv2.IMREAD_GRAYSCALE) for i in sample["id"]]
sample["brightness"] = [im.mean() for im in imgs]
sample["contrast"] = [im.std() for im in imgs]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(sample["brightness"], bins=40, ax=axes[0]).set(title="Mean intensity")
sns.histplot(sample["contrast"], bins=40, ax=axes[1]).set(title="Intensity std (contrast)")
plt.show()
print(sample[["brightness", "contrast"]].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(18, 4))
ext = pd.concat([sample.nsmallest(3, "brightness"), sample.nlargest(3, "brightness")])
for ax, (_, r) in zip(axes, ext.iterrows()):
    ax.imshow(cv2.imread(str(config.TRAIN_IMG_DIR / f"{r.id}.png"), cv2.IMREAD_GRAYSCALE), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"brightness {r.brightness:.0f}"); ax.axis("off")
plt.suptitle("Darkest vs brightest images"); plt.tight_layout(); plt.show()